# Router (xác định dạng bài) trên benchmark Số phức v2

Mô hình: Qwen3-0.6B + LoRA, bản đã đạt 100% (repo slm-highschool-math-tutoring, `models/router_lora`). Benchmark: 2160 câu v2 (repo react-calculator-complex-numbers). Mọi thứ tự tải từ GitHub, không cần Add Data.

**Cách chạy trên mỗi tài khoản:**
1. Settings: Accelerator = **GPU T4** (hoặc P100); **Internet = On** (tài khoản cần xác minh số điện thoại).
2. Ở ô đầu tiên, đổi `SHARD_ID` thành số của tài khoản: 0, 1, 2, 3, 4, 5, 6.
3. Save Version, chọn **Save & Run All (Commit)**.
4. Khi chạy xong, tải `router_v2_shardKof7.csv` và `router_summary_shardKof7.json` trong tab **Output**.


In [ ]:
import os
SHARD_ID = 0            # <-- DOI SO NAY cho moi tai khoan: 0, 1, 2, 3, 4, 5, 6
N_SHARDS = 7            # 2160 cau chia thanh 7 phan (moi phan ~309 cau). Dat 1 neu chay het tren 1 tai khoan
RUN_V1_CONTROL = False  # True: cham them chinh cac cau do o de goc v1 (doi chung, mat gap doi thoi gian)
LIMIT = 0               # >0: chi cham thu so cau nay (kiem tra nhanh)
SHARD_ID = int(os.environ.get('SHARD_ID', SHARD_ID)); N_SHARDS = int(os.environ.get('N_SHARDS', N_SHARDS))
LIMIT = int(os.environ.get('ROUTER_LIMIT', LIMIT)); RUN_V1_CONTROL = os.environ.get('RUN_V1_CONTROL', str(RUN_V1_CONTROL)) == 'True'
assert 0 <= SHARD_ID < N_SHARDS

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('peft') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft'], check=False)

In [ ]:
import json, csv, hashlib, urllib.request, time
SLM = 'https://raw.githubusercontent.com/manhdungcr7/slm-highschool-math-tutoring/65401343c8e50ff0c499d358b1a306e2029e2eaf/'
SPR = 'https://raw.githubusercontent.com/manhdungcr7/react-calculator-complex-numbers/b73e74dbec4282e46305acb8be14c0f72a92ef5d/'
ON_KAGGLE = os.name == 'posix' and os.path.isdir('/kaggle/working')
WORK = '/kaggle/working' if ON_KAGGLE else os.environ.get('ROUTER_OUT', './router_work')
LORA_DIR = os.path.join(WORK, 'router_lora')
os.makedirs(LORA_DIR, exist_ok=True)
FILES = {
    os.path.join(LORA_DIR, 'adapter_config.json'): SLM + 'models/router_lora/adapter_config.json',
    os.path.join(LORA_DIR, 'adapter_model.safetensors'): SLM + 'models/router_lora/adapter_model.safetensors',
    os.path.join(WORK, 'candidates.json'): SLM + 'data/router_train/kaggle_bundle/candidates.json',
    os.path.join(WORK, 'eval_set.jsonl'): SLM + 'data/router_train/kaggle_bundle/eval_set.jsonl',
    os.path.join(WORK, 'benchmark_v2.csv'): SPR + 'data/questions/benchmark_so_phuc_v2_24dang_2160cau.csv',
}
for path, url in FILES.items():
    if not os.path.exists(path):
        for attempt in range(3):
            try:
                urllib.request.urlretrieve(url, path); break
            except Exception as e:
                print('thu lai', url, repr(e)[:80]); time.sleep(5)
    print(f'{os.path.getsize(path):>10,}  {os.path.basename(path)}')

def git_blob_sha(path):
    data = open(path, 'rb').read()
    return hashlib.sha1(b'blob %d\0' % len(data) + data).hexdigest()
# adapter phai dung la ban da dat 100% (cung ma bam git voi models/router_lora tren repo)
assert git_blob_sha(os.path.join(LORA_DIR, 'adapter_model.safetensors')) == '7ca685f36307a552836cc5b23f8b991bfd108180', 'adapter khong dung ban 100%'
print('adapter dung ban da dat 100%')

In [ ]:
PROMPT = ('Hãy xác định dạng bài của đề bài sau. Chỉ trả lời đúng một dòng theo mẫu '
          '"Dạng N: mô tả dạng bài", trong đó N là số thứ tự của dạng.\n\nĐề bài:\n')
CANDS = json.load(open(os.path.join(WORK, 'candidates.json'), encoding='utf-8'))
assert len(CANDS) == 142 and all(c.startswith(f'Dạng {i + 1}:') for i, c in enumerate(CANDS))
label_of = {}
for line in open(os.path.join(WORK, 'eval_set.jsonl'), encoding='utf-8'):
    e = json.loads(line)
    label_of.setdefault(e['prompt'], set()).add(int(e['so_dang']))
bench = list(csv.DictReader(open(os.path.join(WORK, 'benchmark_v2.csv'), encoding='utf-8-sig')))
assert len(bench) == 2160
rows = []
for r in bench:
    lab = label_of.get(PROMPT + r['de_bai_v1'])
    assert lab and len(lab) == 1, f"khong tim thay nhan cho cau {r['stt']}"
    rows.append(dict(stt=int(r['stt']), ma_dang=r['ma_dang_bai_bao'], bien_the=int(r['bien_the']), so_dang=next(iter(lab)),
                     prompt_v2=PROMPT + r['de_bai'], prompt_v1=PROMPT + r['de_bai_v1']))
print('2160/2160 cau co nhan | so dang:', len({r['so_dang'] for r in rows}))
rows = [r for i, r in enumerate(rows) if i % N_SHARDS == SHARD_ID]
if LIMIT:
    rows = rows[::max(1, len(rows) // LIMIT)][:LIMIT]
print(f'phan {SHARD_ID}/{N_SHARDS}: {len(rows)} cau')

In [ ]:
import copy, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
BASE = 'Qwen/Qwen3-0.6B'
EOS = '<|im_end|>'
HAS_CUDA = torch.cuda.is_available()
DTYPE = torch.float16 if HAS_CUDA else torch.float32
tok = AutoTokenizer.from_pretrained(BASE)        # cung tokenizer va chat template voi luc huan luyen (da kiem tra: ma hoa giong het)
try:
    base = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE)
except TypeError:
    base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=DTYPE)
# Kaggle cai san torchao 0.10 (cu); peft moi bao loi khi gap ban cu. Router khong dung torchao -> bao cho peft la khong co torchao.
try:
    import peft.import_utils as _pi, peft.tuners.lora.torchao as _pt
    _pi.is_torchao_available = lambda: False
    _pt.is_torchao_available = lambda: False
except Exception as e:
    print('bo qua va torchao:', repr(e)[:80])
model = PeftModel.from_pretrained(base, LORA_DIR)
if HAS_CUDA:
    model = model.to('cuda')
model.eval(); model.config.use_cache = True
DEV = next(model.parameters()).device
print('GPU:', torch.cuda.get_device_name(0) if HAS_CUDA else 'khong co', '| dtype:', next(model.parameters()).dtype)

def prompt_text(user):
    return tok.apply_chat_template([{'role': 'user', 'content': user}], tokenize=False, add_generation_prompt=True, enable_thinking=False)

CAND_IDS = [tok(c + EOS, add_special_tokens=False).input_ids for c in CANDS]
COLON = tok(':', add_special_tokens=False).input_ids[0]
LABEL_LEN = [ids.index(COLON) + 1 for ids in CAND_IDS]     # so token cua phan "Dạng N:"

In [ ]:
NO_CACHE = False

@torch.no_grad()
def prompt_state(user):
    p_ids = tok(prompt_text(user), add_special_tokens=False, return_tensors='pt').input_ids.to(DEV)
    out = model(input_ids=p_ids, use_cache=True)
    return out.past_key_values, p_ids.shape[1], torch.log_softmax(out.logits[0, -1].float(), -1), p_ids

@torch.no_grad()
def sum_logprob(state, token_lists, chunk=71):
    global NO_CACHE
    cache, L, first, p_ids = state
    res = torch.zeros(len(token_lists))
    for s in range(0, len(token_lists), chunk):
        part = token_lists[s:s + chunk]; n = len(part); m = max(len(x) for x in part)
        ids = torch.full((n, m), tok.pad_token_id, dtype=torch.long); msk = torch.zeros((n, m), dtype=torch.long)
        for i, x in enumerate(part):
            ids[i, :len(x)] = torch.tensor(x); msk[i, :len(x)] = 1
        if not NO_CACHE:
            try:
                c = copy.deepcopy(cache); c.batch_repeat_interleave(n)
                attn = torch.cat([torch.ones((n, L), dtype=torch.long), msk], 1).to(DEV)
                o = model(input_ids=ids.to(DEV), attention_mask=attn, past_key_values=c, use_cache=True)
                logp = torch.log_softmax(o.logits.float(), -1)
                tgt = ids[:, 1:].to(DEV)
                tok_lp = logp[:, :-1].gather(2, tgt.unsqueeze(-1)).squeeze(-1) * msk[:, 1:].to(DEV)
                tok_lp = torch.cat([first[ids[:, 0].to(DEV)].unsqueeze(1), tok_lp], 1)
            except Exception as e:
                print('khong dung duoc cache, cham khong cache:', repr(e)[:100]); NO_CACHE = True
        if NO_CACHE:
            full = torch.cat([p_ids.repeat(n, 1), ids.to(DEV)], 1)
            attn = torch.cat([torch.ones((n, L), dtype=torch.long), msk], 1).to(DEV)
            logp = torch.log_softmax(model(input_ids=full, attention_mask=attn).logits[:, L - 1:-1].float(), -1)
            tok_lp = logp.gather(2, ids.to(DEV).unsqueeze(-1)).squeeze(-1) * msk.to(DEV)
        cum = tok_lp.cumsum(1)
        for i in range(n):
            res[s + i] = cum[i, len(part[i]) - 1].item()
    return res

@torch.no_grad()
def predict(user, top=5):
    state = prompt_state(user)
    label = sum_logprob(state, [ids[:LABEL_LEN[i]] for i, ids in enumerate(CAND_IDS)])
    best = torch.topk(label, top).indices.tolist()
    full = sum_logprob(state, [CAND_IDS[i] for i in best], chunk=top)
    order = [best[j] for j in torch.argsort(full, descending=True).tolist()]
    return dict(pred=order[0] + 1, pred_buoc1=best[0] + 1, top3=' '.join(str(k + 1) for k in order[:3]),
                conf=round(float(torch.softmax(label, 0).max()), 6))

In [ ]:
sets = ['v2'] + (['v1'] if RUN_V1_CONTROL else [])
out_rows = []
OUT = os.path.join(WORK, f'router_v2_shard{SHARD_ID}of{N_SHARDS}.csv')
if os.path.exists(OUT):                               # chay tiep tu cho dung (neu lan truoc bi ngat)
    out_rows = list(csv.DictReader(open(OUT, encoding='utf-8-sig')))
    for o in out_rows:
        for k in list(o):
            if k.startswith('dung_'): o[k] = o[k] == 'True'
            elif k.split('_')[0] in ('stt', 'bien', 'so', 'pred'): o[k] = int(o[k])
    done = {int(o['stt']) for o in out_rows}
    rows = [r for r in rows if r['stt'] not in done]
    print('chay tiep: da co', len(done), 'cau, con', len(rows))
t0 = time.time()
for i, r in enumerate(rows):
    d = dict(stt=r['stt'], ma_dang=r['ma_dang'], bien_the=r['bien_the'], so_dang=r['so_dang'])
    for s in sets:
        p = predict(r['prompt_' + s])
        d.update({f'{k}_{s}': v for k, v in p.items()})
        d[f'dung_{s}'] = p['pred'] == r['so_dang']
    out_rows.append(d)
    if i == 9:
        print(f'toc do: {(time.time() - t0) / 10:.2f} giay/cau -> khoang {(time.time() - t0) / 10 * len(rows) / 60:.0f} phut', flush=True)
    if i % 25 == 24 or i == len(rows) - 1:
        print(f'{len(out_rows)} cau | v2 dung {sum(o["dung_v2"] for o in out_rows)}/{len(out_rows)} | {(time.time() - t0) / 60:.1f} phut', flush=True)
        with open(OUT, 'w', encoding='utf-8-sig', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(out_rows[0])); w.writeheader(); w.writerows(out_rows)

In [ ]:
from collections import defaultdict
n = len(out_rows)
summary = dict(shard=SHARD_ID, n_shards=N_SHARDS, n=n, minutes=round((time.time() - t0) / 60, 1))
for s in sets:
    summary[f'dung_{s}'] = f"{sum(o[f'dung_{s}'] for o in out_rows)}/{n}"
    summary[f'buoc1_dung_{s}'] = f"{sum(o[f'pred_buoc1_{s}'] == o['so_dang'] for o in out_rows)}/{n}"
per = defaultdict(lambda: [0, 0])
for o in out_rows:
    per[o['ma_dang']][0] += o['dung_v2']; per[o['ma_dang']][1] += 1
summary['theo_dang_v2'] = {k: f'{a}/{b}' for k, (a, b) in sorted(per.items())}
summary['cau_sai_v2'] = [dict(stt=o['stt'], dang=o['ma_dang'], dung=o['so_dang'], du_doan=o['pred_v2'], conf=o['conf_v2']) for o in out_rows if not o['dung_v2']]
json.dump(summary, open(os.path.join(WORK, f'router_summary_shard{SHARD_ID}of{N_SHARDS}.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(json.dumps(summary, ensure_ascii=False, indent=1))
for f in os.listdir(LORA_DIR):                       # chi giu ket qua trong Output, xoa ban sao mo hinh va du lieu da tai
    os.remove(os.path.join(LORA_DIR, f))
os.rmdir(LORA_DIR)
for f in ('candidates.json', 'eval_set.jsonl', 'benchmark_v2.csv'):
    os.remove(os.path.join(WORK, f))
print('Ket qua:', OUT)